# FunctionGemma 津波避難の関数呼び出し学習
GPU（T4以上）で実行します。先に Hugging Face で `google/functiongemma-270m-it` の利用条件に同意し、Colab のシークレット `HF_TOKEN` に読み取りトークンを登録してください。
学習用の1,600件から160件を検証に分け、別の400件は最終評価にのみ使います。

In [ ]:
%pip install -q "torch" "transformers==4.57.1" "trl==0.25.1" "datasets==4.4.1" "accelerate" "protobuf" "sentencepiece"

In [ ]:
from google.colab import files, userdata
from pathlib import Path
import os, shutil
uploaded = files.upload()  # train_functiongemma.py, train.jsonl, test.jsonl の3ファイル
required = {'train_functiongemma.py', 'train.jsonl', 'test.jsonl'}
assert required.issubset(uploaded), f'不足: {required - set(uploaded)}'
root = Path('/content/tsunami_finetune')
(root / 'finetune' / 'data').mkdir(parents=True, exist_ok=True)
(root / 'finetune' / '__init__.py').touch()
for name in required:
    destination = root / 'finetune' / ('data/' if name.endswith('.jsonl') else '') / name
    shutil.move(name, destination)
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
assert os.environ['HF_TOKEN'], 'Colab のシークレット HF_TOKEN を設定してください'
print('ファイルを配置しました。GPU:', __import__('torch').cuda.get_device_name(0))

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'finetune.train_functiongemma', '--output', 'finetune/output'], cwd='/content/tsunami_finetune', check=True)
assert (root / 'finetune/output/report.json').exists(), '学習レポートが作成されていません'

In [ ]:
from google.colab import drive
import shutil
assert (root / 'finetune/output/report.json').exists(), '学習が完了していません。前のセルのエラーを確認してください'
drive.mount('/content/drive')
destination = '/content/drive/MyDrive/tsunami-functiongemma-output'
shutil.copytree('/content/tsunami_finetune/finetune/output', destination, dirs_exist_ok=True)
print('保存先:', destination)
print((root / 'finetune/output/report.json').read_text(encoding='utf-8'))